# Chapter 9: Vector Spaces and Subspaces

**Book:** *Linear Algebra with Applications in Machine Learning: From Intuitive Understanding to Python Coding*

---

Vector spaces are the abstract scaffolding on which all of linear algebra rests. A vector space is any collection of objects that can be added together and scaled while staying inside the collection. Subspaces are "sub-playgrounds" that inherit this structure. This notebook covers:

1. **Vector Space Axioms** -- verifying axioms in $\mathbb{R}^n$
2. **Subspaces** -- testing the three criteria (zero vector, closure under addition, closure under scaling)
3. **Null Space** -- $\text{Null}(A) = \{\mathbf{x} \mid A\mathbf{x} = \mathbf{0}\}$, Rank-Nullity Theorem
4. **Column and Row Spaces** -- bases via RREF, Rank Theorem
5. **Basis and Dimension** -- minimal spanning sets, linear independence
6. **Coordinate Systems** -- representing vectors relative to a basis $\mathcal{B}$
7. **ML Applications** -- PCA as subspace projection, feature independence

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import sympy as sp
from scipy.linalg import null_space

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

print("All imports successful.")

## 9.1 Vector Spaces: Definition and Basics

A **vector space** $V$ over $\mathbb{R}$ is a set equipped with addition and scalar multiplication satisfying eight axioms (commutativity, associativity, identities, inverses, distributivity, compatibility). The most familiar example is $\mathbb{R}^n$.

### Verifying Axioms in $\mathbb{R}^3$

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
w = np.array([-1, 0, 2])
a, b = 3.0, -2.0
zero = np.zeros(3)

axioms = [
    ("Commutativity: u+v = v+u",
     np.allclose(u + v, v + u)),
    ("Associativity: (u+v)+w = u+(v+w)",
     np.allclose((u + v) + w, u + (v + w))),
    ("Additive identity: v + 0 = v",
     np.allclose(v + zero, v)),
    ("Additive inverse: v + (-v) = 0",
     np.allclose(v + (-v), zero)),
    ("Scalar compat: a(bv) = (ab)v",
     np.allclose(a * (b * v), (a * b) * v)),
    ("Scalar identity: 1*v = v",
     np.allclose(1 * v, v)),
    ("Distrib (vec): a(u+v) = au+av",
     np.allclose(a * (u + v), a * u + a * v)),
    ("Distrib (scalar): (a+b)v = av+bv",
     np.allclose((a + b) * v, a * v + b * v)),
]

print("Verifying vector space axioms for R^3:")
for desc, result in axioms:
    print(f"  {desc:45s} {'PASS' if result else 'FAIL'}")
print(f"\nAll axioms satisfied: {all(r for _, r in axioms)}")

In [ ]:
# Concrete examples
print(f"u = {u}")
print(f"v = {v}")
print(f"u + v = {u + v}")
print(f"2 * u  = {2 * u}")
print(f"Zero vector: {zero}")
print(f"Additive inverse of v: {-v}")

## 9.2 Subspaces

A subset $V \subseteq \mathbb{R}^n$ is a **subspace** if it satisfies three conditions:

1. $\mathbf{0} \in V$ (contains the zero vector)
2. $\mathbf{u}, \mathbf{v} \in V \Rightarrow \mathbf{u} + \mathbf{v} \in V$ (closed under addition)
3. $c \in \mathbb{R},\ \mathbf{u} \in V \Rightarrow c\mathbf{u} \in V$ (closed under scalar multiplication)

**Key insight:** Any set defined as $\text{span}\{\mathbf{v}_1, \dots, \mathbf{v}_k\}$ is automatically a subspace.

In [ ]:
# Example: U = { [2x-y, 3x, x+y]^T | x,y in R } is a subspace
# Because it equals span{[2,3,1], [-1,0,1]}

g1 = np.array([2, 3, 1])   # coefficient of x
g2 = np.array([-1, 0, 1])  # coefficient of y

print("U = span{ [2,3,1], [-1,0,1] }")

# 1. Zero vector: x=0, y=0
print(f"\n1. Zero vector (x=0, y=0): {0*g1 + 0*g2}  => in U")

# 2. Closure under addition
u1 = 1*g1 + 2*g2  # x=1, y=2
u2 = 3*g1 + (-1)*g2  # x=3, y=-1
u_sum = u1 + u2
# u_sum should equal (1+3)*g1 + (2+(-1))*g2 = 4*g1 + 1*g2
print(f"2. u1 = {u1}, u2 = {u2}")
print(f"   u1 + u2 = {u_sum}")
print(f"   4*g1 + 1*g2 = {4*g1 + 1*g2}  => matches, closed under addition")

# 3. Closure under scalar multiplication
c = -3
print(f"3. c*u1 = {c}*{u1} = {c*u1}")
print(f"   {c}*1*g1 + {c}*2*g2 = {c*1*g1 + c*2*g2}  => matches, closed under scaling")

### Non-Subspace: The Right Half-Plane

The set $S = \{[x, y]^T \in \mathbb{R}^2 \mid x \geq 0\}$ is NOT a subspace because it fails closure under scalar multiplication.

In [ ]:
# Demonstrate failure
u = np.array([2, 3])  # in S (x >= 0)
c = -1
cu = c * u
print(f"u = {u} is in S (x={u[0]} >= 0)")
print(f"c*u = {c}*{u} = {cu}")
print(f"c*u has x = {cu[0]} < 0 => NOT in S!")
print("=> S is NOT a subspace (fails closure under scalar multiplication)")

### Subspace: Plane Through the Origin

The set $W = \{\mathbf{w} \in \mathbb{R}^3 \mid 6w_1 - 5w_2 + 4w_3 = 0\}$ is a subspace (a plane containing the origin).

In [ ]:
# Visualize the subspace plane 6w1 - 5w2 + 4w3 = 0 => w1 = (5w2 - 4w3)/6
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

w2, w3 = np.meshgrid(np.linspace(-2, 2, 30), np.linspace(-2, 2, 30))
w1 = (5 * w2 - 4 * w3) / 6

ax.plot_surface(w1, w2, w3, alpha=0.4, cmap='viridis')

# Mark the origin
ax.scatter([0], [0], [0], color='red', s=80, zorder=5, label='Origin')

# Normal vector [6, -5, 4]
n = np.array([6, -5, 4]) / np.linalg.norm([6, -5, 4]) * 1.5
ax.quiver(0, 0, 0, n[0], n[1], n[2], color='red', linewidth=2.5,
          arrow_length_ratio=0.12, label='Normal [6,-5,4]')

ax.set_xlabel(r'$w_1$')
ax.set_ylabel(r'$w_2$')
ax.set_zlabel(r'$w_3$')
ax.set_title(r'Subspace: $6w_1 - 5w_2 + 4w_3 = 0$', fontsize=13)
ax.legend()
ax.view_init(elev=25, azim=40)
plt.tight_layout()
plt.show()

## 9.3 Null Space

The **null space** (kernel) of $A \in \mathbb{R}^{m \times n}$ is:

$$\text{Null}(A) = \{\mathbf{x} \in \mathbb{R}^n \mid A\mathbf{x} = \mathbf{0}\}$$

It is always a subspace of $\mathbb{R}^n$ (the domain). Its dimension is the **nullity**, connected to rank by the **Rank-Nullity Theorem**:

$$\text{rank}(A) + \text{nullity}(A) = n$$

In [ ]:
# Check null space membership
A = np.array([[2, 5, 1],
              [-1, -7, -5],
              [3, 4, -2]])

u = np.array([1, 0, 4])
v = np.array([2, -1, 1])

print(f"A @ u = {A @ u}  =>  {'IN' if np.allclose(A @ u, 0) else 'NOT IN'} Null(A)")
print(f"A @ v = {A @ v}  =>  {'IN' if np.allclose(A @ v, 0) else 'NOT IN'} Null(A)")

### Computing Null Space via RREF (SymPy)

In [ ]:
# Example: A (3x4) with nullity 2
A = sp.Matrix([[1, 2, 1, -1],
               [2, 4, 0, -8],
               [0, 0, 2, 6]])

print("A =")
sp.pprint(A)

# RREF
rref, pivots = A.rref()
print("\nRREF =")
sp.pprint(rref)
print(f"Pivot columns: {pivots}")

# Null space
ns = A.nullspace()
print(f"\nNull space basis ({len(ns)} vector(s)):")
for i, v in enumerate(ns):
    print(f"  n{i+1} = {v.T}")

# Rank-Nullity
rank = A.rank()
nullity = len(ns)
n = A.cols
print(f"\nRank = {rank}, Nullity = {nullity}, n = {n}")
print(f"Rank + Nullity = {rank + nullity} = n? {rank + nullity == n}")

# Verify
for i, v in enumerate(ns):
    print(f"A * n{i+1} = {(A * v).T}")

In [ ]:
# SciPy null_space for numerical computation
A_np = np.array([[1, 2, 1, -1],
                 [2, 4, 0, -8],
                 [0, 0, 2, 6]], dtype=float)

ns_scipy = null_space(A_np)
print(f"Null space basis (scipy), shape {ns_scipy.shape}:")
print(ns_scipy.round(4))
print(f"\nDimension (nullity): {ns_scipy.shape[1]}")
print(f"Verification: A @ ns =\n{(A_np @ ns_scipy).round(10)}  (should be all zeros)")

### Rank-Nullity Theorem in Action

In [ ]:
# Test Rank-Nullity on several matrices
matrices = [
    ("Full rank 3x3", np.array([[1,0,0],[0,1,0],[0,0,1]], dtype=float)),
    ("Rank-deficient 3x4", np.array([[1,2,1,-1],[2,4,0,-8],[0,0,2,6]], dtype=float)),
    ("Rank 1 (3x4)", np.array([[1,2,3,4],[2,4,6,8],[3,6,9,12]], dtype=float)),
    ("Wide 2x5", np.array([[3,6,6,3,9],[6,12,13,0,3]], dtype=float)),
]

print(f"{'Name':25s} {'m x n':>6s} {'rank':>5s} {'nullity':>8s} {'r+n=n?':>7s}")
print("-" * 55)
for name, M in matrices:
    m, n = M.shape
    r = np.linalg.matrix_rank(M)
    ns = null_space(M)
    nul = ns.shape[1]
    ok = (r + nul == n)
    print(f"{name:25s} {m}x{n:>2d} {r:5d} {nul:8d} {'YES' if ok else 'NO':>7s}")

## 9.4 Column Space and Row Space

- **Column space** $\text{Col}(A)$: span of columns of $A$, a subspace of $\mathbb{R}^m$ (the output space). Its dimension is the rank.
- **Row space** $\text{Row}(A)$: span of rows of $A$, a subspace of $\mathbb{R}^n$ (the input space). Same dimension as column space.

The **Rank Theorem** states: $\dim(\text{Col}(A)) = \dim(\text{Row}(A)) = \text{rank}(A)$.

In [ ]:
A = sp.Matrix([[1, 2, 3],
               [4, 5, 6]])

print("A =")
sp.pprint(A)

# RREF to identify pivot columns
rref, pivots = A.rref()
print("\nRREF =")
sp.pprint(rref)
print(f"Pivot columns: {pivots}")

# Column space basis: original columns at pivot positions
col_basis = [A.col(j) for j in pivots]
print(f"\nColumn space basis (pivot columns from original A):")
for v in col_basis:
    print(f"  {v.T}")

# Row space basis: nonzero rows of RREF
row_basis = [rref.row(i) for i in range(rref.rows) if not rref.row(i).is_zero_matrix]
print(f"\nRow space basis (nonzero rows of RREF):")
for v in row_basis:
    print(f"  {v}")

print(f"\nRank Theorem: dim(Col) = dim(Row) = {A.rank()}")

In [ ]:
# NumPy: column space, row space, null space all at once
def four_fundamental_subspaces(A_np, name='A'):
    """Compute dimensions of the four fundamental subspaces."""
    m, n = A_np.shape
    r = np.linalg.matrix_rank(A_np)
    ns_A = null_space(A_np)
    ns_AT = null_space(A_np.T)
    
    print(f"=== {name} ({m}x{n}), rank = {r} ===")
    print(f"  Column space:     dim = {r}        (subspace of R^{m})")
    print(f"  Row space:        dim = {r}        (subspace of R^{n})")
    print(f"  Null space:       dim = {ns_A.shape[1]}        (subspace of R^{n})")
    print(f"  Left null space:  dim = {ns_AT.shape[1]}        (subspace of R^{m})")
    print(f"  rank + nullity = {r} + {ns_A.shape[1]} = {r + ns_A.shape[1]} = n = {n}")
    print(f"  rank + left-null = {r} + {ns_AT.shape[1]} = {r + ns_AT.shape[1]} = m = {m}")
    print()

four_fundamental_subspaces(np.array([[1,2,3],[4,5,6]], dtype=float), 'A (2x3)')
four_fundamental_subspaces(np.array([[1,2,1,-1],[2,4,0,-8],[0,0,2,6]], dtype=float), 'B (3x4)')

## 9.5 Basis and Dimension

A **basis** for a subspace $V$ is a set of vectors that is both linearly independent and spanning. The number of vectors in any basis is the **dimension** of $V$.

**Dimension Theorem:** All bases for a given vector space have the same number of vectors.

In [ ]:
# Basis for V = { x in R^3 | x1 + x2 + x3 = 0 }
# x1 = -x2 - x3, so x = x2*[-1,1,0] + x3*[-1,0,1]

b1 = np.array([-1, 1, 0])
b2 = np.array([-1, 0, 1])

print("Subspace V: x1 + x2 + x3 = 0")
print(f"Basis: b1 = {b1}, b2 = {b2}")

# Check linear independence
B = np.column_stack([b1, b2])
print(f"Rank of [b1|b2] = {np.linalg.matrix_rank(B)} (= 2 => independent)")

# Check spanning: any v in V should be expressible as c1*b1 + c2*b2
v_test = np.array([-3, 1, 2])  # -3 + 1 + 2 = 0, so v in V
coords = np.linalg.lstsq(B, v_test, rcond=None)[0]
print(f"\nTest: v = {v_test} (sum = {v_test.sum()})")
print(f"Coordinates: c1={coords[0]:.2f}, c2={coords[1]:.2f}")
print(f"Reconstruction: {coords[0]*b1 + coords[1]*b2}")
print(f"\nDimension of V = {np.linalg.matrix_rank(B)}")

In [ ]:
# Dimension Theorem: different bases, same number of vectors
# Basis 1 for R^2
basis1 = [np.array([1, 0]), np.array([0, 1])]
# Basis 2 for R^2
basis2 = [np.array([1, 1]), np.array([1, -1])]

print(f"Basis 1 for R^2: {[list(b) for b in basis1]}, size = {len(basis1)}")
print(f"Basis 2 for R^2: {[list(b) for b in basis2]}, size = {len(basis2)}")
print(f"Both have {len(basis1)} vectors => dimension of R^2 is 2")

### Finding a Basis for a Span (Removing Redundant Vectors)

In [ ]:
# Span of {[1,2,3], [2,4,6], [1,1,1]} -- second vector is 2x the first
vecs = sp.Matrix([[1, 2, 3],
                  [2, 4, 6],
                  [1, 1, 1]]).T  # columns are the vectors

print("Vectors (as columns):")
sp.pprint(vecs)

rref_v, pivots_v = vecs.rref()
print(f"\nRREF pivot columns: {pivots_v}")
print(f"Basis for the span: columns {list(pivots_v)} of the original matrix")
for j in pivots_v:
    print(f"  {vecs.col(j).T}")
print(f"Dimension = {len(pivots_v)}")

### Nullity Example: Rank-1 Matrix

In [ ]:
# A (3x4) where all rows are multiples of the first
A = sp.Matrix([[1, 2, 3, 4],
               [2, 4, 6, 8],
               [3, 6, 9, 12]])

print("A =")
sp.pprint(A)

ns = A.nullspace()
print(f"\nRank = {A.rank()}, Nullity = {len(ns)}, n = {A.cols}")
print(f"Rank + Nullity = {A.rank() + len(ns)} = n? {A.rank() + len(ns) == A.cols}")

print(f"\nNull space basis ({len(ns)} vectors):")
for i, v in enumerate(ns):
    print(f"  n{i+1} = {v.T}")

## 9.6 Coordinate Systems

Given a basis $\mathcal{B} = \{\mathbf{b}_1, \dots, \mathbf{b}_k\}$ for a subspace, every vector $\mathbf{v}$ in the subspace has a unique coordinate representation:

$$\mathbf{v} = c_1 \mathbf{b}_1 + \dots + c_k \mathbf{b}_k, \quad [\mathbf{v}]_{\mathcal{B}} = \begin{bmatrix} c_1 \\ \vdots \\ c_k \end{bmatrix}$$

The coordinates are found by solving $B \mathbf{c} = \mathbf{v}$, where $B$ has the basis vectors as columns.

In [ ]:
# 2D example: B = {[1,0], [1,1]}
B_mat = np.array([[1, 1],
                  [0, 1]])  # columns are basis vectors
v = np.array([3, 2])

coords = np.linalg.solve(B_mat, v)
print(f"Basis B = {{[1,0], [1,1]}}")
print(f"v = {v}")
print(f"[v]_B = {coords}")
print(f"Verify: {coords[0]}*[1,0] + {coords[1]}*[1,1] = {coords[0]*np.array([1,0]) + coords[1]*np.array([1,1])}")

In [ ]:
# 3D example: find coordinates and convert back
B_mat3 = np.array([[1, 0, 1],
                   [1, 1, 0],
                   [0, 1, 1]])  # columns = basis vectors
v3 = np.array([0, 1, 3])

coords3 = np.linalg.solve(B_mat3, v3)
print(f"Basis B = {{ [1,1,0], [0,1,1], [1,0,1] }}")
print(f"v = {v3}")
print(f"[v]_B = {coords3}")
print(f"Verify: B @ [v]_B = {B_mat3 @ coords3}")

In [ ]:
# Going the other direction: given coordinates, find the standard vector
B_other = np.array([[1, 1, 1],
                    [1, -1, 2],
                    [1, 1, 2]])  # columns = basis vectors
v_B = np.array([3, 6, -2])  # coordinates in basis B

v_std = B_other @ v_B
print(f"Coordinates [v]_B = {v_B}")
print(f"v in standard basis = B @ [v]_B = {v_std}")

### Visualizing a Change of Basis in 2D

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

v = np.array([3, 2])
b1 = np.array([1, 0])
b2 = np.array([1, 1])
B_mat = np.column_stack([b1, b2])
coords = np.linalg.solve(B_mat, v)

# Standard basis view
ax1.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
           color='red', linewidth=2.5, label=f'v = {v}')
ax1.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
           color='blue', linewidth=2, label='e1 = [1,0]')
ax1.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
           color='green', linewidth=2, label='e2 = [0,1]')
ax1.set_title('Standard Basis\n[v]_std = (3, 2)', fontsize=12)
ax1.set_xlim(-0.5, 4)
ax1.set_ylim(-0.5, 3)
ax1.set_aspect('equal')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Custom basis view
ax2.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
           color='red', linewidth=2.5, label=f'v = {v}')
ax2.quiver(0, 0, b1[0], b1[1], angles='xy', scale_units='xy', scale=1,
           color='blue', linewidth=2, label=f'b1 = {b1}')
ax2.quiver(0, 0, b2[0], b2[1], angles='xy', scale_units='xy', scale=1,
           color='green', linewidth=2, label=f'b2 = {b2}')
# Show decomposition
ax2.quiver(0, 0, coords[0]*b1[0], coords[0]*b1[1],
           angles='xy', scale_units='xy', scale=1,
           color='blue', linewidth=1.5, alpha=0.4)
ax2.quiver(coords[0]*b1[0], coords[0]*b1[1],
           coords[1]*b2[0], coords[1]*b2[1],
           angles='xy', scale_units='xy', scale=1,
           color='green', linewidth=1.5, alpha=0.4)

ax2.set_title(f'Custom Basis B\n[v]_B = ({coords[0]:.0f}, {coords[1]:.0f})', fontsize=12)
ax2.set_xlim(-0.5, 4)
ax2.set_ylim(-0.5, 3)
ax2.set_aspect('equal')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Same Vector, Different Coordinate Systems', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.7 ML Applications

### 9.7.1 PCA as Subspace Projection

PCA finds a low-dimensional subspace (spanned by eigenvectors of the covariance matrix) that captures the most variance. Projecting data onto this subspace reduces dimensionality while preserving structure.

In [ ]:
np.random.seed(42)
cov = np.array([[3, 2], [2, 2]])
data = np.random.multivariate_normal([0, 0], cov, 200)

# Compute covariance and PCA basis
C = np.cov(data.T)
eigvals, eigvecs = np.linalg.eigh(C)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

pc1 = eigvecs[:, 0]  # first principal component

# Project onto 1D subspace (span of PC1)
projections = data @ pc1[:, None] @ pc1[None, :]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(data[:, 0], data[:, 1], alpha=0.3, s=15, color='steelblue', label='Data')
ax.scatter(projections[:, 0], projections[:, 1], alpha=0.3, s=15, color='red', label='Projected (1D)')

# Draw PC1 direction
scale = 3 * np.sqrt(eigvals[0])
ax.annotate('', xy=scale*pc1, xytext=-scale*pc1,
            arrowprops=dict(arrowstyle='-', color='red', lw=2))
ax.text(*(scale*pc1 * 1.1), f'PC1 ({eigvals[0]/eigvals.sum()*100:.0f}% var)',
        fontsize=11, fontweight='bold', color='red')

ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('PCA: Projecting Data onto a 1D Subspace', fontsize=13)
ax.set_aspect('equal')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 9.7.2 Feature Independence via Rank

The rank of the feature matrix tells us how many truly independent features we have. If rank < number of columns, some features are redundant (linearly dependent), and the null space captures those dependencies.

In [ ]:
# Redundant features: col3 = col1 + col2
X = np.array([[1, 2, 3],
              [4, 5, 9],
              [2, 1, 3],
              [0, 3, 3]], dtype=float)

rank = np.linalg.matrix_rank(X)
ns = null_space(X)

print(f"Feature matrix X ({X.shape[0]} samples, {X.shape[1]} features):")
print(X)
print(f"\nRank = {rank} (only {rank} independent features)")
print(f"Nullity = {ns.shape[1]} (redundancy direction)")
if ns.shape[1] > 0:
    dep = ns[:, 0] / ns[:, 0][np.argmax(np.abs(ns[:, 0]))]  # normalize
    print(f"Null space vector (dependency): {dep.round(4)}")
    print(f"This says: {dep[0]:.1f}*col1 + {dep[1]:.1f}*col2 + {dep[2]:.1f}*col3 = 0")

## 9.8 Exercises

Selected exercises from the chapter.

**Exercise 3:** Find a basis and dimension for $\text{Null}(A)$ where $A = \begin{bmatrix} 1 & 2 & 1 \\ 2 & 4 & 2 \end{bmatrix}$.

In [ ]:
# Exercise 3: Your code here


**Exercise 8:** Compute $\text{Null}(A)$ and its dimension for $A = \begin{bmatrix} 1 & 2 & 3 \\ 0 & 1 & 2 \end{bmatrix}$.

In [ ]:
# Exercise 8: Your code here


**Exercise 15:** Find the coordinate vector of $\mathbf{v} = [2, 3]^T$ with respect to $\mathcal{B} = \{[2,1]^T, [-1,1]^T\}$.

In [ ]:
# Exercise 15: Your code here


**Exercise 20:** Compute the null space of $A = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$ and verify its dimension.

In [ ]:
# Exercise 20: Your code here


---

## Exercise Solutions

In [ ]:
# --- Solution: Exercise 3 ---
A = sp.Matrix([[1, 2, 1], [2, 4, 2]])
ns = A.nullspace()
print(f"Null space basis ({len(ns)} vectors):")
for v in ns:
    print(f"  {v.T}")
print(f"Nullity = {len(ns)}")
print(f"Rank = {A.rank()}, Rank + Nullity = {A.rank() + len(ns)} = n = {A.cols}")

In [ ]:
# --- Solution: Exercise 8 ---
A = sp.Matrix([[1, 2, 3], [0, 1, 2]])
ns = A.nullspace()
print(f"Null space basis:")
for v in ns:
    print(f"  {v.T}")
print(f"Dimension (nullity) = {len(ns)}")

In [ ]:
# --- Solution: Exercise 15 ---
B = np.array([[2, -1], [1, 1]], dtype=float)
v = np.array([2, 3], dtype=float)
coords = np.linalg.solve(B, v)
print(f"[v]_B = {coords}")
print(f"Verify: {coords[0]}*[2,1] + {coords[1]}*[-1,1] = {B @ coords}")

In [ ]:
# --- Solution: Exercise 20 ---
A = np.array([[1, 2], [2, 4]], dtype=float)
ns = null_space(A)
print(f"Null space basis:\n{ns.round(4)}")
print(f"Dimension = {ns.shape[1]}")
print(f"Rank = {np.linalg.matrix_rank(A)}, Rank + Nullity = {np.linalg.matrix_rank(A) + ns.shape[1]} = n = {A.shape[1]}")
print(f"Verify: A @ ns =\n{(A @ ns).round(10)}")